# Skill Graph Building

This notebook builds a skill relationship graph from co-occurrence patterns in job postings, resumes, and learning pathways. The graph captures skill relationships including:

- **Prerequisites**: Skills typically learned before others
- **Co-skills**: Skills frequently appearing together
- **Similar Skills**: Skills with overlapping contexts
- **Skill Hierarchies**: Parent-child relationships (e.g., Python → Programming)

## Pipeline Flow
- Input: Canonical roles and skills from `workspace.gold.canonical_roles`, `workspace.gold.skill_catalog`
- Output: Skill graph edges written to `workspace.gold.skill_graph`
- Metrics: Graph statistics in `workspace.gold.skill_graph_metrics`

## Graph Types
1. **Co-occurrence Graph**: Skills appearing together in job postings
2. **Prerequisite Graph**: Directional relationships based on skill progression
3. **Similarity Graph**: Skills with semantic similarity

In [0]:
%pip install networkx
%restart_python

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType, TimestampType
from pyspark.sql.window import Window
from datetime import datetime
import networkx as nx
import matplotlib.pyplot as plt

# Configuration
CONFIG = {
    "skill_catalog_table": "workspace.gold.skill_catalog",
    "job_skills_table": "workspace.silver.job_posting_skills",  # Job ID + Skill list
    "skill_graph_table": "workspace.gold.skill_graph",
    "graph_metrics_table": "workspace.gold.skill_graph_metrics",
    "min_cooccurrence_count": 5,  # Minimum times skills appear together
    "min_support": 0.01,  # Minimum support for association rules
    "edge_types": ["cooccurrence", "prerequisite", "similar"],
    "max_edges_per_skill": 20  # Limit edges to keep graph manageable
}

print("Skill Graph Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [0]:
# Load canonical skill catalog
try:
    skill_catalog_df = spark.table(CONFIG["skill_catalog_table"]).select(
        "skill_id",
        "canonical_skill_name",
        "skill_category"
    )
    skill_count = skill_catalog_df.count()
    print(f"Loaded {skill_count} skills from catalog")
    display(skill_catalog_df.limit(10))
except Exception as e:
    print(f"Warning: Could not load skill catalog: {e}")
    print("Creating sample skill catalog...")
    
    sample_skills = [
        ("skill_0000000001", "Python", "Programming Language"),
        ("skill_0000000002", "Java", "Programming Language"),
        ("skill_0000000003", "SQL", "Database"),
        ("skill_0000000004", "Machine Learning", "AI/ML"),
        ("skill_0000000005", "Deep Learning", "AI/ML"),
        ("skill_0000000006", "Data Analysis", "Analytics"),
        ("skill_0000000007", "Statistics", "Analytics"),
        ("skill_0000000008", "Communication", "Soft Skill"),
    ]
    
    skill_catalog_df = spark.createDataFrame(
        sample_skills,
        ["skill_id", "canonical_skill_name", "skill_category"]
    )
    skill_count = skill_catalog_df.count()
    print(f"Sample catalog created with {skill_count} skills")

In [0]:
# Load job postings and extract skills from titles/descriptions
print("Loading jobs from workspace.silver.silver_jobs_current...")

# Load active jobs
jobs_df = spark.table("workspace.silver.silver_jobs_current").filter(
    (F.col("is_active") == True) & 
    (F.col("soft_delete_flag") == False)
).select(
    F.col("enterprise_job_id").alias("job_id"),
    F.coalesce(F.col("title_normalized"), F.col("title_raw")).alias("title"),
    F.col("description_raw").alias("description")
).limit(1000)  # Limit for performance

print(f"Loaded {jobs_df.count()} jobs from silver layer")

# Collect skill catalog for keyword matching
skill_keywords = skill_catalog_df.select(
    "skill_id",
    F.lower("canonical_skill_name").alias("skill_name_lower")
).collect()

print(f"Matching {len(skill_keywords)} skills against job content...")

# Extract skills from job text using keyword matching
from pyspark.sql.types import ArrayType

def extract_skills_from_text(title, description, skill_map):
    """Extract skill IDs from job title and description via keyword matching."""
    if title is None and description is None:
        return []
    
    combined_text = (title or "").lower() + " " + (description or "").lower()
    matched_skills = set()
    
    for skill_id, skill_name in skill_map:
        # Simple keyword matching - check if skill name appears in text
        if skill_name in combined_text:
            matched_skills.add(skill_id)
    
    return list(matched_skills)

# Broadcast skill keywords for efficiency
skill_map = [(row.skill_id, row.skill_name_lower) for row in skill_keywords]

# Register UDF
extract_skills_udf = F.udf(
    lambda title, desc: extract_skills_from_text(title, desc, skill_map),
    ArrayType(StringType())
)

# Apply skill extraction
job_skills_df = jobs_df.withColumn(
    "skill_ids",
    extract_skills_udf(F.col("title"), F.col("description"))
).filter(
    F.size(F.col("skill_ids")) > 0  # Keep only jobs with at least 1 matched skill
).select(
    "job_id",
    "skill_ids"
)

job_count = job_skills_df.count()
print(f"Extracted skills from {job_count} jobs")

if job_count == 0:
    print("\nWarning: No skills matched in job data. Using sample data instead...")
    
    # Fallback to sample data if no matches
    sample_jobs = [
        ("job_001", ["skill_0000000001", "skill_0000000003", "skill_0000000004"]),  # Python, SQL, ML
        ("job_002", ["skill_0000000001", "skill_0000000004", "skill_0000000006"]),  # Python, ML, Analysis
        ("job_003", ["skill_0000000002", "skill_0000000003"]),  # Java, SQL
        ("job_004", ["skill_0000000001", "skill_0000000006", "skill_0000000007"]),  # Python, Analysis, Stats
        ("job_005", ["skill_0000000004", "skill_0000000005", "skill_0000000001"]),  # ML, DL, Python
        ("job_006", ["skill_0000000001", "skill_0000000003", "skill_0000000006"]),  # Python, SQL, Analysis
        ("job_007", ["skill_0000000002", "skill_0000000003", "skill_0000000008"]),  # Java, SQL, Communication
        ("job_008", ["skill_0000000001", "skill_0000000004", "skill_0000000007"]),  # Python, ML, Stats
    ]
    
    job_skills_df = spark.createDataFrame(
        sample_jobs,
        StructType([
            StructField("job_id", StringType(), False),
            StructField("skill_ids", ArrayType(StringType()), False)
        ])
    )
    job_count = job_skills_df.count()
    print(f"Sample data created with {job_count} job postings")

print(f"\nJob-skill associations ready: {job_count} records")
display(job_skills_df.limit(10))

In [0]:
# Explode skill arrays into pairs for co-occurrence analysis
from pyspark.sql.functions import explode, col, array

# Self-join to create skill pairs from same job posting
job_skills_exploded = job_skills_df.select(
    col("job_id"),
    explode(col("skill_ids")).alias("skill_id")
)

# Create pairs (exclude self-pairs)
skill_pairs_df = job_skills_exploded.alias("a").join(
    job_skills_exploded.alias("b"),
    (col("a.job_id") == col("b.job_id")) & (col("a.skill_id") < col("b.skill_id")),
    "inner"
).select(
    col("a.job_id"),
    col("a.skill_id").alias("skill_1"),
    col("b.skill_id").alias("skill_2")
)

print(f"Generated {skill_pairs_df.count()} skill pairs from job postings")
display(skill_pairs_df.limit(10))

In [0]:
# Count co-occurrences
cooccurrence_df = skill_pairs_df.groupBy("skill_1", "skill_2").agg(
    F.count("*").alias("cooccurrence_count")
).filter(
    F.col("cooccurrence_count") >= CONFIG["min_cooccurrence_count"]
)

print(f"Found {cooccurrence_df.count()} skill pairs with >= {CONFIG['min_cooccurrence_count']} co-occurrences")

# Calculate support (proportion of jobs containing both skills)
total_jobs = job_count

cooccurrence_df = cooccurrence_df.withColumn(
    "support",
    F.col("cooccurrence_count") / total_jobs
).filter(
    F.col("support") >= CONFIG["min_support"]
)

print(f"After support filtering (>= {CONFIG['min_support']}): {cooccurrence_df.count()} pairs")

# Add skill names for readability
cooccurrence_with_names_df = cooccurrence_df \
    .join(skill_catalog_df.select("skill_id", "canonical_skill_name"), 
          cooccurrence_df.skill_1 == skill_catalog_df.skill_id) \
    .withColumnRenamed("canonical_skill_name", "skill_1_name") \
    .drop("skill_id") \
    .join(skill_catalog_df.select("skill_id", "canonical_skill_name"),
          cooccurrence_df.skill_2 == skill_catalog_df.skill_id) \
    .withColumnRenamed("canonical_skill_name", "skill_2_name") \
    .drop("skill_id")

print("\nTop Co-occurring Skills:")
display(cooccurrence_with_names_df.orderBy(F.desc("cooccurrence_count")).limit(20))

In [0]:
# Create bidirectional edges for co-occurrence graph
cooccurrence_edges_df = cooccurrence_df.select(
    F.col("skill_1").alias("source_skill_id"),
    F.col("skill_2").alias("target_skill_id"),
    F.lit("cooccurrence").alias("edge_type"),
    F.col("support").alias("weight"),
    F.col("cooccurrence_count").alias("frequency"),
    F.current_timestamp().alias("created_timestamp")
)

# Add reverse edges for undirected graph
reverse_edges_df = cooccurrence_edges_df.select(
    F.col("target_skill_id").alias("source_skill_id"),
    F.col("source_skill_id").alias("target_skill_id"),
    F.col("edge_type"),
    F.col("weight"),
    F.col("frequency"),
    F.col("created_timestamp")
)

all_cooccurrence_edges_df = cooccurrence_edges_df.union(reverse_edges_df)

print(f"Created {all_cooccurrence_edges_df.count()} co-occurrence edges (bidirectional)")
display(all_cooccurrence_edges_df.limit(10))

In [0]:
# Build prerequisite relationships using heuristics
# Example: "Deep Learning" requires "Machine Learning" as prerequisite

# Define prerequisite rules based on skill categories and known relationships
prerequisite_rules = [
    ("skill_0000000005", "skill_0000000004", 0.9),  # Deep Learning -> Machine Learning
    ("skill_0000000004", "skill_0000000007", 0.8),  # Machine Learning -> Statistics
    ("skill_0000000004", "skill_0000000001", 0.85), # Machine Learning -> Python
    ("skill_0000000006", "skill_0000000003", 0.75), # Data Analysis -> SQL
]

if prerequisite_rules:
    prerequisite_edges_df = spark.createDataFrame(
        prerequisite_rules,
        ["source_skill_id", "target_skill_id", "weight"]
    ).withColumn(
        "edge_type",
        F.lit("prerequisite")
    ).withColumn(
        "frequency",
        F.lit(None).cast(IntegerType())
    ).withColumn(
        "created_timestamp",
        F.current_timestamp()
    )
    
    print(f"Created {prerequisite_edges_df.count()} prerequisite edges")
    display(prerequisite_edges_df)
else:
    prerequisite_edges_df = spark.createDataFrame([], StructType([
        StructField("source_skill_id", StringType()),
        StructField("target_skill_id", StringType()),
        StructField("edge_type", StringType()),
        StructField("weight", FloatType()),
        StructField("frequency", IntegerType()),
        StructField("created_timestamp", TimestampType())
    ]))
    print("No prerequisite rules defined")

In [0]:
# Build similarity edges based on shared categories or contexts
# Skills in the same category are similar

similarity_edges = []

# Group skills by category
category_skills = skill_catalog_df.groupBy("skill_category").agg(
    F.collect_list("skill_id").alias("skill_ids")
).collect()

for row in category_skills:
    category = row["skill_category"]
    skills = row["skill_ids"]
    
    # Create pairs within same category
    for i in range(len(skills)):
        for j in range(i + 1, len(skills)):
            similarity_edges.append((
                skills[i],
                skills[j],
                "similar",
                0.7,  # Base similarity weight for same category
                None
            ))

if similarity_edges:
    from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType, TimestampType
    
    schema = StructType([
        StructField("source_skill_id", StringType(), True),
        StructField("target_skill_id", StringType(), True),
        StructField("edge_type", StringType(), True),
        StructField("weight", FloatType(), True),
        StructField("frequency", IntegerType(), True)
    ])
    
    similarity_edges_df = spark.createDataFrame(
        similarity_edges,
        schema
    ).withColumn(
        "created_timestamp",
        F.current_timestamp()
    )
    
    # Make bidirectional
    reverse_similarity_df = similarity_edges_df.select(
        F.col("target_skill_id").alias("source_skill_id"),
        F.col("source_skill_id").alias("target_skill_id"),
        F.col("edge_type"),
        F.col("weight"),
        F.col("frequency"),
        F.col("created_timestamp")
    )
    
    all_similarity_edges_df = similarity_edges_df.union(reverse_similarity_df)
    
    print(f"Created {all_similarity_edges_df.count()} similarity edges (bidirectional)")
    display(all_similarity_edges_df.limit(10))
else:
    all_similarity_edges_df = spark.createDataFrame([], StructType([
        StructField("source_skill_id", StringType()),
        StructField("target_skill_id", StringType()),
        StructField("edge_type", StringType()),
        StructField("weight", FloatType()),
        StructField("frequency", IntegerType()),
        StructField("created_timestamp", TimestampType())
    ]))
    print("No similarity edges created")

In [0]:
# Combine all edge types into unified skill graph
# Standardize schemas before union (column order and types)
standard_columns = [
    "source_skill_id", "target_skill_id", "edge_type", "weight", "frequency", "created_timestamp"
]

# Reorder and cast columns to match
all_cooccurrence_standardized = all_cooccurrence_edges_df.select(
    *[F.col(c).cast("string") if c in ["source_skill_id", "target_skill_id", "edge_type"] 
      else F.col(c).cast("double") if c == "weight"
      else F.col(c).cast("bigint") if c == "frequency"
      else F.col(c) for c in standard_columns]
)

prerequisite_standardized = prerequisite_edges_df.select(
    *[F.col(c).cast("string") if c in ["source_skill_id", "target_skill_id", "edge_type"] 
      else F.col(c).cast("double") if c == "weight"
      else F.col(c).cast("bigint") if c == "frequency"
      else F.col(c) for c in standard_columns]
)

all_similarity_standardized = all_similarity_edges_df.select(
    *[F.col(c).cast("string") if c in ["source_skill_id", "target_skill_id", "edge_type"] 
      else F.col(c).cast("double") if c == "weight"
      else F.col(c).cast("bigint") if c == "frequency"
      else F.col(c) for c in standard_columns]
)

all_edges_df = all_cooccurrence_standardized \
    .union(prerequisite_standardized) \
    .union(all_similarity_standardized)

total_edges = all_edges_df.count()
print(f"\nTotal skill graph edges: {total_edges}")

# Show edge type distribution
edge_stats_df = all_edges_df.groupBy("edge_type").agg(
    F.count("*").alias("edge_count"),
    F.avg("weight").alias("avg_weight")
).orderBy(F.desc("edge_count"))

print("\nEdge Type Distribution:")
display(edge_stats_df)

In [0]:
# Limit number of edges per skill to keep graph manageable
# Keep top N edges by weight for each source skill

window_spec = Window.partitionBy("source_skill_id", "edge_type").orderBy(F.desc("weight"))

pruned_edges_df = all_edges_df.withColumn(
    "edge_rank",
    F.row_number().over(window_spec)
).filter(
    F.col("edge_rank") <= CONFIG["max_edges_per_skill"]
).drop("edge_rank")

pruned_count = pruned_edges_df.count()
print(f"\nAfter pruning (max {CONFIG['max_edges_per_skill']} edges per skill/type): {pruned_count} edges")
print(f"Removed {total_edges - pruned_count} edges ({(total_edges - pruned_count)/total_edges*100:.1f}%)")

In [0]:
# Write skill graph edges
print(f"\nWriting {pruned_count} edges to {CONFIG['skill_graph_table']}...")

pruned_edges_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(CONFIG["skill_graph_table"])

print("✓ Skill graph written successfully")

In [0]:
# Calculate per-skill graph metrics
skill_metrics_df = pruned_edges_df.groupBy("source_skill_id").agg(
    F.count("*").alias("out_degree"),
    F.sum(F.when(F.col("edge_type") == "cooccurrence", 1).otherwise(0)).alias("cooccurrence_edges"),
    F.sum(F.when(F.col("edge_type") == "prerequisite", 1).otherwise(0)).alias("prerequisite_edges"),
    F.sum(F.when(F.col("edge_type") == "similar", 1).otherwise(0)).alias("similarity_edges"),
    F.avg("weight").alias("avg_edge_weight"),
    F.max("weight").alias("max_edge_weight")
)

# Join with skill names
skill_metrics_with_names_df = skill_metrics_df.join(
    skill_catalog_df.select("skill_id", "canonical_skill_name", "skill_category"),
    skill_metrics_df.source_skill_id == skill_catalog_df.skill_id,
    "left"
).select(
    "source_skill_id",
    "canonical_skill_name",
    "skill_category",
    "out_degree",
    "cooccurrence_edges",
    "prerequisite_edges",
    "similarity_edges",
    "avg_edge_weight",
    "max_edge_weight"
).withColumn(
    "calculated_timestamp",
    F.current_timestamp()
)

print("\nSkill Graph Metrics:")
display(skill_metrics_with_names_df.orderBy(F.desc("out_degree")).limit(20))

# Write metrics
print(f"\nWriting metrics to {CONFIG['graph_metrics_table']}...")
skill_metrics_with_names_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(CONFIG["graph_metrics_table"])

print("✓ Graph metrics written successfully")

In [0]:
print("\n" + "="*60)
print("SKILL GRAPH BUILD - EXECUTION SUMMARY")
print("="*60)
print(f"Input skills: {skill_count}")
print(f"Input job postings: {job_count}")
print(f"Total edges created: {total_edges}")
print(f"Edges after pruning: {pruned_count}")
print(f"Max edges per skill: {CONFIG['max_edges_per_skill']}")
print(f"\nEdge types created:")
for row in edge_stats_df.collect():
    print(f"  {row['edge_type']}: {row['edge_count']} edges (avg weight: {row['avg_weight']:.3f})")
print("="*60)